In [ ]:
import copy
import csv
import os
import warnings
from argparse import ArgumentParser
import numpy as np
import torch
from tqdm import tqdm
import yaml
from torch.utils import data
# 개별 json 라벨 파일을 이용해 학습 데이터 리스트 생성
import glob
import json
import os
import pandas as pd
from PIL import Image
from torch.utils import data
import numpy as np
import cv2
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:",device)

In [ ]:
# M1 TCGA-only 데이터 로드
# 입력 단위: 1 slide(case)의 tile image paths + tile 좌표 + slide 전체 크기 + multi-horizon survival label/mask
# 출력 label: dead by 6/12/18/24 months. Unknown(censored before horizon)은 mask=0으로 loss에서 제외합니다.

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"

M1_OUTPUT_PATH = Path("outputs/M1")
M1_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

DATASET_NAMES = ["TCGA_PAAD", "CPTAC_PDAC"]
DATASET_NAME = "TCGA_CPTAC"
SEED = 42
PATCH_INPUT_SIZE = 512  # 1.0 MPP 1024 tile을 512 입력으로 줄이면 effective MPP는 약 2.0입니다.
PRELOAD_RESIZED_TILES = True  # dataset 구성 단계에서 512x512 tile을 CPU memory에 preload합니다.
PRELOAD_TILE_SIZE = PATCH_INPUT_SIZE
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)
TEST_SIZE = 0.2
VALID_SIZE = 0.25  # train_valid 내부 비율. 전체 기준 0.8 * 0.25 = 0.2
REQUIRE_COMPLETE_24M_HORIZONS = True

MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


def load_clinical_label(dataset: str, case_id: str) -> tuple[float | None, int | None]:
    clinical_json_path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not clinical_json_path.exists():
        return None, None
    with open(clinical_json_path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    clinical = clinical_json.get("clinical", {})
    return clinical.get("os_time_days"), clinical.get("os_event")


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    """Create multi-horizon dead-by labels and known-label mask.

    label[h] = 1 if death occurred by horizon h, else 0.
    mask[h] = 1 if the label at horizon h is known.

    Censored before a horizon is unknown and excluded from BCE loss with mask=0.
    """
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask

    os_time_days = float(os_time_days)
    os_event = int(os_event)

    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            # Censored before this horizon: unknown.
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def get_patch_padding(image_size: int = PATCH_INPUT_SIZE) -> tuple[int, int, int, int]:
    # UNI2-h는 patch_size=14라 512 입력이 바로 들어갈 수 없습니다.
    # 512로 먼저 resize한 뒤 가장 가까운 patch-size multiple까지 symmetric padding합니다.
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    target_size = int(np.ceil(image_size / patch_size) * patch_size)
    pad_total = max(0, target_size - image_size)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return (pad_left, pad_left, pad_right, pad_right)


def get_model_input_size(image_size: int = PATCH_INPUT_SIZE) -> int:
    patch_size = int(globals().get("FEATURE_EXTRACTOR_PATCH_SIZE", 16))
    return int(np.ceil(image_size / patch_size) * patch_size)


def get_train_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    # Resize를 augmentation보다 먼저 수행해 1024 tile에서 바로 연산하지 않도록 합니다.
    # 이후 patch-size multiple로 padding합니다. UNI2-h는 512 -> 518 padding이 필요합니다.
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_patch_transform(image_size: int = PATCH_INPUT_SIZE):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_train_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    # dataset 구성 단계에서 이미 512x512로 resize된 image에 대해 padding/augmentation/normalization만 수행합니다.
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomApply([transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.5),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def get_eval_cached_patch_transform(image_size: int = PRELOAD_TILE_SIZE):
    return transforms.Compose([
        transforms.Pad(get_patch_padding(image_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def estimate_tile_cache_gb(n_tiles: int, image_size: int = PRELOAD_TILE_SIZE) -> float:
    return n_tiles * image_size * image_size * 3 / (1024 ** 3)


def preload_resized_tile_images(tile_index: pd.DataFrame, image_size: int = PRELOAD_TILE_SIZE) -> dict[str, np.ndarray]:
    unique_paths = sorted(tile_index["tile_path"].astype(str).unique())
    expected_gb = estimate_tile_cache_gb(len(unique_paths), image_size=image_size)
    print(f"Preloading resized tiles: {len(unique_paths):,} tiles, {image_size}x{image_size}, expected uint8 memory ~{expected_gb:.2f} GB")

    cache = {}
    resize_size = (image_size, image_size)
    for tile_path in tqdm(unique_paths, desc="Preload resized tile images", unit="tile"):
        with Image.open(tile_path) as image:
            image = image.convert("RGB")
            image = image.resize(resize_size, resample=Image.BILINEAR)
            cache[tile_path] = np.asarray(image, dtype=np.uint8).copy()
    return cache


def add_tile_coordinates(tile_df: pd.DataFrame) -> pd.DataFrame:
    tile_df = tile_df.copy()
    if {"x_level0", "y_level0"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_level0"].astype(int)
        tile_df["y"] = tile_df["y_level0"].astype(int)
    elif {"x_total_matrix", "y_total_matrix"}.issubset(tile_df.columns):
        tile_df["x"] = tile_df["x_total_matrix"].astype(int)
        tile_df["y"] = tile_df["y_total_matrix"].astype(int)
    else:
        raise ValueError("tile metadata must contain x/y coordinate columns.")
    return tile_df


def get_slide_dimensions(tile_df: pd.DataFrame) -> tuple[int, int]:
    source_size = tile_df["source_tile_size_px"].astype(float)
    width = int((tile_df["x"].astype(float) + source_size).max())
    height = int((tile_df["y"].astype(float) + source_size).max())
    return width, height


def load_case_tiles(dataset: str, metadata_path: Path) -> tuple[pd.DataFrame, dict]:
    case_id = metadata_path.parent.name
    case_dir = metadata_path.parent
    tile_df = pd.read_csv(metadata_path)

    tile_df = add_tile_coordinates(tile_df)
    slide_width, slide_height = get_slide_dimensions(tile_df)

    tile_df["tile_path"] = tile_df["tile_path"].astype(str)
    tile_df = tile_df[tile_df["tile_path"].map(lambda x: Path(x).exists())].copy()
    tile_df["slide_width"] = slide_width
    tile_df["slide_height"] = slide_height

    source_size = tile_df["source_tile_size_px"].astype(float)
    tile_df["x_norm"] = tile_df["x"].astype(float) / slide_width
    tile_df["y_norm"] = tile_df["y"].astype(float) / slide_height
    tile_df["x_center_norm"] = (tile_df["x"].astype(float) + source_size / 2) / slide_width
    tile_df["y_center_norm"] = (tile_df["y"].astype(float) + source_size / 2) / slide_height
    tile_df["w_norm"] = source_size / slide_width
    tile_df["h_norm"] = source_size / slide_height

    os_time, os_event = load_clinical_label(dataset, case_id)
    if os_time is None or os_event is None:
        os_time = tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan
        os_event = tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan

    y, mask = make_horizon_label_mask(os_time, os_event)
    case_record = {
        "dataset": dataset,
        "slide_uid": f"{dataset}::{case_id}",
        "case_id": case_id,
        "case_dir": case_dir.as_posix(),
        "metadata_path": metadata_path.as_posix(),
        "n_tiles": int(len(tile_df)),
        "slide_width": slide_width,
        "slide_height": slide_height,
        "os_time_days": float(os_time) if pd.notna(os_time) else np.nan,
        "os_event": int(os_event) if pd.notna(os_event) else np.nan,
        "known_horizon_count": int(mask.sum()),
    }
    for i, name in enumerate(HORIZON_NAMES):
        case_record[name] = float(y[i])
        case_record[f"mask_{name}"] = float(mask[i])
    tile_df["dataset"] = dataset
    tile_df["case_id"] = case_id
    tile_df["slide_uid"] = case_record["slide_uid"]
    return tile_df, case_record


case_records = []
tile_index_list = []
for dataset in DATASET_NAMES:
    metadata_paths = sorted((IMAGE_PATH / dataset).glob("*/tile_metadata.csv"))
    for metadata_path in tqdm(metadata_paths, desc=f"Load {dataset} metadata"):
        tile_df, case_record = load_case_tiles(dataset, metadata_path)
        if case_record["n_tiles"] == 0:
            case_record["exclude_reason"] = "no_tiles"
            case_records.append(case_record)
            continue
        tile_index_list.append(tile_df)
        case_records.append(case_record)

all_slide_df = pd.DataFrame(case_records)
tile_index_df = pd.concat(tile_index_list, ignore_index=True)


complete_horizon_mask = all_slide_df[[f"mask_{name}" for name in HORIZON_NAMES]].eq(1).all(axis=1)
required_horizon_mask = complete_horizon_mask if REQUIRE_COMPLETE_24M_HORIZONS else all_slide_df["known_horizon_count"].gt(0)

eligible_mask = (
    all_slide_df["os_time_days"].notna()
    & all_slide_df["os_event"].notna()
    & required_horizon_mask
    & all_slide_df["n_tiles"].gt(0)
)
excluded_df = all_slide_df[~eligible_mask].copy()
if not excluded_df.empty:
    excluded_df["exclude_reason"] = np.select(
        [
            excluded_df["n_tiles"].le(0),
            excluded_df["os_time_days"].isna(),
            excluded_df["os_event"].isna(),
            ~required_horizon_mask.loc[excluded_df.index],
        ],
        ["no_tiles", "missing_os_time", "missing_os_event", "incomplete_required_horizon"],
        default="unknown",
    )

slide_df = all_slide_df[eligible_mask].copy()
slide_df["os_event"] = slide_df["os_event"].astype(int)
tile_index_df = tile_index_df[tile_index_df["slide_uid"].isin(slide_df["slide_uid"])].copy()

slide_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_cptac_horizon_slide_manifest.csv", index=False)
tile_index_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_cptac_horizon_tile_index.csv", index=False)
excluded_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_cptac_horizon_excluded_cases.csv", index=False)

print("all_slide_df:", all_slide_df.shape)
print("complete 24m horizon cases:", int(complete_horizon_mask.sum()))
print("slide_df:", slide_df.shape)
print("excluded_df:", excluded_df.shape)
print("tile_index_df:", tile_index_df.shape)

if PRELOAD_RESIZED_TILES:
    if "TILE_IMAGE_CACHE" not in globals() or not TILE_IMAGE_CACHE:
        TILE_IMAGE_CACHE = preload_resized_tile_images(tile_index_df, image_size=PRELOAD_TILE_SIZE)
    else:
        print(f"Using existing TILE_IMAGE_CACHE: {len(TILE_IMAGE_CACHE):,} tiles")
else:
    TILE_IMAGE_CACHE = {}

print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("cached tiles:", len(TILE_IMAGE_CACHE))

horizon_summary = []
for name in HORIZON_NAMES:
    mask_col = f"mask_{name}"
    known = slide_df[mask_col].eq(1)
    horizon_summary.append({
        "horizon": name,
        "known_n": int(known.sum()),
        "dead_n": int(slide_df.loc[known, name].sum()),
        "alive_n": int(known.sum() - slide_df.loc[known, name].sum()),
        "unknown_n": int((~known).sum()),
        "dead_rate_known": float(slide_df.loc[known, name].mean()) if known.any() else np.nan,
    })
horizon_summary_df = pd.DataFrame(horizon_summary)
display(horizon_summary_df)
display(slide_df[["case_id", "n_tiles", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].head())


class M1SlideDataset(Dataset):
    """TCGA case-level pathology dataset for multi-horizon MIL training."""

    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            slide_uid: group.sort_values(["y", "x"]).reset_index(drop=True)
            for slide_uid, group in tile_index.groupby("slide_uid")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["slide_uid"]]

        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        label = row[HORIZON_NAMES].to_numpy(np.float32)
        slide_size = np.array([row["slide_width"], row["slide_height"]], dtype=np.float32)

        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "slide_uid": row["slide_uid"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "slide_size": torch.from_numpy(slide_size),
            "horizon_months": torch.tensor(HORIZON_MONTHS, dtype=torch.float32),
            "label": torch.from_numpy(label),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
        }


class M1TileDataset(Dataset):
    """Tile-level dataset for UNI/UNI v2 on-the-fly feature extraction with augmentation."""

    def __init__(self, tile_index: pd.DataFrame, transform=None, tile_cache: dict[str, np.ndarray] | None = None):
        self.tile_index = tile_index.reset_index(drop=True).copy()
        self.transform = transform
        self.tile_cache = tile_cache or {}

    def __len__(self):
        return len(self.tile_index)

    def __getitem__(self, idx):
        row = self.tile_index.iloc[idx]
        tile_path = row["tile_path"]
        if tile_path in self.tile_cache:
            image = Image.fromarray(self.tile_cache[tile_path])
        else:
            with Image.open(tile_path) as image_file:
                image = image_file.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        coords = row[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        return {
            "image": image,
            "coords": torch.from_numpy(coords),
            "case_id": row["case_id"],
            "slide_uid": row["slide_uid"],
            "tile_path": tile_path,
        }


# 24개월까지 모든 horizon label이 확인 가능한 TCGA case만 사용하고, 전체 기준 6:2:2로 split합니다.
slide_df["stratify_group"] = slide_df["dataset"].astype(str) + "_event" + slide_df["os_event"].astype(str)
stratify_for_test = slide_df["stratify_group"] if slide_df["stratify_group"].value_counts().min() >= 2 else slide_df["os_event"]
train_valid_df, test_df = train_test_split(
    slide_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=stratify_for_test,
)
stratify_for_valid = train_valid_df["stratify_group"] if train_valid_df["stratify_group"].value_counts().min() >= 2 else train_valid_df["os_event"]
train_df, valid_df = train_test_split(
    train_valid_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=stratify_for_valid,
)

train_case_ids = set(train_df["slide_uid"])
valid_case_ids = set(valid_df["slide_uid"])
test_case_ids = set(test_df["slide_uid"])
assert len(train_case_ids & valid_case_ids) == 0
assert len(train_case_ids & test_case_ids) == 0
assert len(valid_case_ids & test_case_ids) == 0

train_dataset = M1SlideDataset(train_df, tile_index_df)
valid_dataset = M1SlideDataset(valid_df, tile_index_df)
test_dataset = M1SlideDataset(test_df, tile_index_df)

tile_train_transform = get_train_cached_patch_transform() if TILE_IMAGE_CACHE else get_train_patch_transform()
tile_eval_transform = get_eval_cached_patch_transform() if TILE_IMAGE_CACHE else get_eval_patch_transform()

train_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(train_case_ids)],
    transform=tile_train_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
valid_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(valid_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)
test_tile_dataset = M1TileDataset(
    tile_index_df[tile_index_df["slide_uid"].isin(test_case_ids)],
    transform=tile_eval_transform,
    tile_cache=TILE_IMAGE_CACHE,
)

split_df = slide_df[["dataset", "slide_uid", "case_id", "os_time_days", "os_event", "known_horizon_count"] + HORIZON_NAMES + [f"mask_{n}" for n in HORIZON_NAMES]].copy()
split_df["split"] = "unused"
split_df.loc[split_df["slide_uid"].isin(train_case_ids), "split"] = "train"
split_df.loc[split_df["slide_uid"].isin(valid_case_ids), "split"] = "valid"
split_df.loc[split_df["slide_uid"].isin(test_case_ids), "split"] = "test"
split_df.to_csv(M1_OUTPUT_PATH / "m1_tcga_cptac_horizon_case_splits.csv", index=False)

print("slide splits:", len(train_dataset), len(valid_dataset), len(test_dataset))
print("tile splits:", len(train_tile_dataset), len(valid_tile_dataset), len(test_tile_dataset))
print("split x dataset")
display(pd.crosstab(split_df["split"], split_df["dataset"]))
print("split x os_event")
display(pd.crosstab(split_df["split"], split_df["os_event"]))
print("split x dataset x os_event")
display(pd.crosstab([split_df["split"], split_df["dataset"]], split_df["os_event"]))

split_horizon_summary = []
for split_name, part in split_df.groupby("split"):
    row = {"split": split_name, "n_cases": len(part)}
    for name in HORIZON_NAMES:
        known = part[f"mask_{name}"].eq(1)
        row[f"{name}_known"] = int(known.sum())
        row[f"{name}_dead"] = int(part.loc[known, name].sum())
    split_horizon_summary.append(row)
display(pd.DataFrame(split_horizon_summary))

sample = train_dataset[0]
print("sample case:", sample["case_id"])
print("n_tiles:", len(sample["tile_paths"]))
print("coords:", sample["coords"].shape, sample["coords"][:3])
print("slide_size:", sample["slide_size"])
print("label:", sample["label"])
print("model input size:", get_model_input_size(PATCH_INPUT_SIZE), "padding:", get_patch_padding(PATCH_INPUT_SIZE))
print("model output 해석: logits -> sigmoid(logits) * 100 = horizon별 사망위험 percent")


In [ ]:
# M1 모델 정의 및 loss/optimizer 구성
# 구조: frozen UNI/UNI v2 feature extractor + coordinate spatial embedding + attention MIL

import torch
from torch import nn
import pandas as pd
import timm
from huggingface_hub import hf_hub_download

from scripts.models.discrete_survival import (
    harrell_c_index,
    horizon_auc_metrics,
)

from scripts.models.m1_pathology_mil import (
    M1ModelConfig,
    PathologySpatialMIL,
    sample_tiles,
    build_optimizer,
)

if "device" not in globals():
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

MAX_TILES_PER_SLIDE = 512
FEATURE_BATCH_SIZE = 64
SPATIAL_DIM = 128
FUSION_DIM = 128
MIL_HIDDEN_DIM = 64
USE_SPATIAL_EMBEDDING = False
POOLING_MODE = "risk_topk"
DROPOUT = 0.50
N_OUTPUTS = len(HORIZON_NAMES)  # multi-horizon BCE outputs
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-3

# "UNI" 또는 "UNI2-h" 중 선택하세요.
# UNI/UNI2-h는 Hugging Face 접근 권한 또는 로컬 캐시가 필요할 수 있습니다.
FEATURE_EXTRACTOR_NAME = "UNI2-h"
UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}


def load_uni_feature_extractor(
    name: str = FEATURE_EXTRACTOR_NAME,
    device: torch.device = device,
    local_files_only: bool = False,
) -> tuple[nn.Module, int]:
    if name not in UNI_BACKBONES:
        raise ValueError(f"지원하지 않는 feature extractor입니다: {name}. choices={list(UNI_BACKBONES)}")

    cfg = UNI_BACKBONES[name]
    print(f"Loading {name}: {cfg['repo_id']} / {cfg['timm_kwargs']['model_name']}")

    model = timm.create_model(
        pretrained=False,
        **cfg["timm_kwargs"],
    )

    try:
        weight_path = hf_hub_download(
            repo_id=cfg["repo_id"],
            filename=cfg["filename"],
            local_files_only=local_files_only,
        )
    except Exception as exc:
        raise RuntimeError(
            f"{name} weight를 Hugging Face에서 가져오지 못했습니다. "
            "접근 권한/로그인 토큰 또는 네트워크/캐시를 확인하세요. "
            f"repo_id={cfg['repo_id']}, filename={cfg['filename']}"
        ) from exc

    state_dict = torch.load(weight_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("load_state_dict warning")
        print("missing keys:", len(missing))
        print("unexpected keys:", len(unexpected))

    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    model = model.to(device)

    feature_dim = int(getattr(model, "num_features", cfg["feature_dim"]))
    print(f"{name} loaded. feature_dim={feature_dim}")
    return model, feature_dim


# 셀 3에서 UNI/UNI v2까지 로드하고 바로 M1 model/optimizer를 구성합니다.
feature_extractor, M1_FEATURE_DIM = load_uni_feature_extractor(
    name=FEATURE_EXTRACTOR_NAME,
    device=device,
    local_files_only=False,
)
FEATURE_EXTRACTOR_PATCH_SIZE = int(UNI_BACKBONES[FEATURE_EXTRACTOR_NAME]["timm_kwargs"]["patch_size"])
print("FEATURE_EXTRACTOR_PATCH_SIZE:", FEATURE_EXTRACTOR_PATCH_SIZE)
print("PATCH_INPUT_SIZE:", PATCH_INPUT_SIZE, "-> model input size:", get_model_input_size(PATCH_INPUT_SIZE))
print("PATCH_PADDING:", get_patch_padding(PATCH_INPUT_SIZE))

# 2번 셀에서 만든 tile-level dataset transform은 feature extractor patch size를 알기 전 생성되므로 여기서 갱신합니다.
# resized tile cache가 있으면 512 resize를 반복하지 않고 padding/augmentation/normalization만 수행합니다.
if bool(globals().get("TILE_IMAGE_CACHE", {})):
    train_tile_dataset.transform = get_train_cached_patch_transform()
    valid_tile_dataset.transform = get_eval_cached_patch_transform()
    test_tile_dataset.transform = get_eval_cached_patch_transform()
else:
    train_tile_dataset.transform = get_train_patch_transform()
    valid_tile_dataset.transform = get_eval_patch_transform()
    test_tile_dataset.transform = get_eval_patch_transform()

m1_config = M1ModelConfig(
    feature_dim=M1_FEATURE_DIM,
    coord_dim=6,
    spatial_dim=SPATIAL_DIM,
    fusion_dim=FUSION_DIM,
    mil_hidden_dim=MIL_HIDDEN_DIM,
    n_outputs=N_OUTPUTS,
    dropout=DROPOUT,
    max_tiles=MAX_TILES_PER_SLIDE,
    feature_batch_size=FEATURE_BATCH_SIZE,
    freeze_feature_extractor=True,
    use_spatial_embedding=USE_SPATIAL_EMBEDDING,
    pooling_mode=POOLING_MODE,
)


def compute_horizon_pos_weight(train_manifest: pd.DataFrame, horizon_names: list[str], device: torch.device) -> torch.Tensor:
    """BCEWithLogitsLoss용 horizon별 positive class weight 계산.

    complete-24m cohort에서는 모든 horizon label이 관측 가능하므로 mask를 사용하지 않습니다.
    pos_weight = n_negative / n_positive
    """
    rows = []
    weights = []
    for name in horizon_names:
        labels = train_manifest[name].astype(float)
        pos = float(labels.sum())
        neg = float(len(labels) - pos)
        weight = neg / max(pos, 1.0)
        weights.append(weight)
        rows.append({
            "horizon": name,
            "n_cases": int(len(labels)),
            "positive_dead": int(pos),
            "negative_alive": int(neg),
            "pos_weight": weight,
        })
    pos_weight_df = pd.DataFrame(rows)
    display(pos_weight_df)
    return torch.tensor(weights, dtype=torch.float32, device=device)


pos_weight = compute_horizon_pos_weight(train_df, HORIZON_NAMES, device)
bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


def m1_loss_fn(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    return bce_loss_fn(logits.float(), labels.float())

def initialize_m1_model(
    feature_extractor: nn.Module,
    feature_dim: int = M1_FEATURE_DIM,
    lr: float = LEARNING_RATE,
    weight_decay: float = WEIGHT_DECAY,
) -> tuple[PathologySpatialMIL, torch.optim.Optimizer]:
    config = M1ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
        use_spatial_embedding=USE_SPATIAL_EMBEDDING,
        pooling_mode=POOLING_MODE,
    )
    model = PathologySpatialMIL(feature_extractor=feature_extractor, config=config).to(device)
    optimizer = build_optimizer(model, lr=lr, weight_decay=weight_decay)
    return model, optimizer


model, optimizer = initialize_m1_model(feature_extractor, feature_dim=M1_FEATURE_DIM)

print("M1 model initialized.")
print("FEATURE_EXTRACTOR_NAME:", FEATURE_EXTRACTOR_NAME)
print("M1_FEATURE_DIM:", M1_FEATURE_DIM)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("MAX_TILES_PER_SLIDE:", MAX_TILES_PER_SLIDE)
print("FEATURE_BATCH_SIZE:", FEATURE_BATCH_SIZE)
print("USE_SPATIAL_EMBEDDING:", USE_SPATIAL_EMBEDDING)
print("POOLING_MODE:", POOLING_MODE)
print("N_OUTPUTS:", N_OUTPUTS, HORIZON_NAMES)

# train loop에서 사용할 loss 예시:
# outputs = model(tile_images, coords)
# loss = m1_loss_fn(outputs["logits"], labels)
# hazard_percent = outputs["hazard_percent"]
# dead_probability_percent = outputs["hazard_percent"]


In [ ]:
# M1 학습 하이퍼파라미터 정의

from pathlib import Path
import time
import json

MODEL_PATH = Path("../../model")
M1_MODEL_DIR = MODEL_PATH / "pancreatic_cancer_pathology" / "M1" / FEATURE_EXTRACTOR_NAME
M1_MODEL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 50
PATIENCE = 15
MIN_DELTA = 1e-4
MONITOR_METRIC = "valid_mean_auroc"
MONITOR_MODE = "max"
GRAD_CLIP_NORM = 1.0
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

SLIDE_BATCH_SIZE = 1
LOG_EVERY_EPOCHS = 10
SAVE_EVERY_EPOCHS = 10


BEST_CHECKPOINT_PATH = M1_MODEL_DIR / "best_checkpoint.pt"
LAST_CHECKPOINT_PATH = M1_MODEL_DIR / "last_checkpoint.pt"
TRAIN_LOG_PATH = M1_MODEL_DIR / "training_log.csv"
CONFIG_PATH = M1_MODEL_DIR / "training_config.json"

training_config = {
    "dataset": DATASET_NAME,
    "feature_extractor": FEATURE_EXTRACTOR_NAME,
    "patch_input_size": PATCH_INPUT_SIZE,
    "model_input_size": get_model_input_size(PATCH_INPUT_SIZE),
    "preload_resized_tiles": PRELOAD_RESIZED_TILES,
    "preload_tile_size": PRELOAD_TILE_SIZE,
    "patch_padding": get_patch_padding(PATCH_INPUT_SIZE),
    "feature_extractor_patch_size": FEATURE_EXTRACTOR_PATCH_SIZE,
    "effective_mpp": 1.0 * 1024 / PATCH_INPUT_SIZE,
    "max_tiles_per_slide": MAX_TILES_PER_SLIDE,
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "horizon_months": HORIZON_MONTHS,
    "horizon_names": HORIZON_NAMES,
    "loss_function": "multi_horizon_bce_with_logits",
    "output_interpretation": "dead_by_horizon_logits",
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "monitor_metric": MONITOR_METRIC,
    "monitor_mode": MONITOR_MODE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout": DROPOUT,
    "spatial_dim": SPATIAL_DIM,
    "fusion_dim": FUSION_DIM,
    "mil_hidden_dim": MIL_HIDDEN_DIM,
    "use_spatial_embedding": USE_SPATIAL_EMBEDDING,
    "pooling_mode": POOLING_MODE,
    "use_amp": USE_AMP,
    "amp_dtype": str(AMP_DTYPE),
    "seed": SEED,
    "model_dir": M1_MODEL_DIR.as_posix(),
}
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=2, ensure_ascii=False)

# Scheduler는 validation mean AUROC 기준으로 learning rate를 조정합니다.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=MONITOR_MODE,
    factor=0.5,
    patience=5,
    min_lr=1e-6,
)

print("M1_MODEL_DIR:", M1_MODEL_DIR)
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("EPOCHS:", EPOCHS)
print("PATIENCE:", PATIENCE)
print("USE_AMP:", USE_AMP)
print("PRELOAD_RESIZED_TILES:", PRELOAD_RESIZED_TILES)
print("PRELOAD_TILE_SIZE:", PRELOAD_TILE_SIZE)
print("scheduler:", scheduler.__class__.__name__)
print("MONITOR_METRIC:", MONITOR_METRIC, MONITOR_MODE)


In [ ]:
# M1 학습 및 validation
# multi-horizon BCE loss, C-index 보조 지표, horizon-wise AUROC/AUPRC, checkpoint, scheduler

from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def load_tile_tensor_batch(tile_paths: list[str], transform, device: torch.device) -> torch.Tensor:
    images = []
    use_cache = bool(globals().get("TILE_IMAGE_CACHE", {}))
    for path in tile_paths:
        if use_cache and path in TILE_IMAGE_CACHE:
            image = Image.fromarray(TILE_IMAGE_CACHE[path])
        else:
            with Image.open(path) as image_file:
                image = image_file.convert("RGB")
        images.append(transform(image))
    return torch.stack(images, dim=0).to(device, non_blocking=True)


def prepare_slide_batch(sample: dict, training: bool) -> dict:
    if bool(globals().get("TILE_IMAGE_CACHE", {})):
        transform = get_train_cached_patch_transform() if training else get_eval_cached_patch_transform()
    else:
        transform = get_train_patch_transform() if training else get_eval_patch_transform()
    selected_paths, selected_coords, selected_indices = sample_tiles(
        sample["tile_paths"],
        sample["coords"],
        max_tiles=MAX_TILES_PER_SLIDE,
        training=training,
    )
    tile_images = load_tile_tensor_batch(selected_paths, transform=transform, device=device)
    return {
        "tile_images": tile_images,
        "coords": selected_coords.to(device, non_blocking=True),
        "labels": sample["label"].unsqueeze(0).to(device, non_blocking=True).float(),
        "os_time_days": sample["os_time_days"].reshape(1).to(device, non_blocking=True).float(),
        "os_event": sample["os_event"].reshape(1).to(device, non_blocking=True).long(),
        "case_id": sample["case_id"],
        "n_tiles": len(selected_paths),
    }


def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def compute_epoch_metrics(logits: list[np.ndarray], times: list[float], events: list[int], labels: list[np.ndarray]) -> dict:
    if not logits:
        return {"c_index": float("nan"), "accuracy": float("nan")}
    logit_array = np.vstack(logits).astype(float)
    label_array = np.vstack(labels).astype(float)
    score_array = sigmoid_np(logit_array)
    known_mask = np.ones_like(label_array, dtype=bool)
    pred_array = (score_array >= 0.5).astype(float)
    metrics = {
        "c_index": harrell_c_index(times, events, logit_array[:, -1]),
        "accuracy": float((pred_array == label_array).mean()),
    }
    for idx, name in enumerate(HORIZON_NAMES):
        metrics[f"{name}_accuracy"] = float((pred_array[:, idx] == label_array[:, idx]).mean())
    metrics.update(horizon_auc_metrics(label_array, known_mask, score_array, HORIZON_NAMES))
    return metrics


def run_one_epoch(dataset, training: bool, epoch: int) -> dict:
    model.train(training)
    total_loss = 0.0
    total_batches = 0
    all_logits = []
    all_labels = []
    all_times = []
    all_events = []

    iterator = range(len(dataset))
    progress = tqdm(iterator, desc=f"Epoch {epoch:03d} {'train' if training else 'valid'}", leave=False)
    for idx in progress:
        sample = dataset[idx]
        batch = prepare_slide_batch(sample, training=training)

        if training:
            optimizer.zero_grad(set_to_none=True)

        if USE_AMP:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                outputs = model(batch["tile_images"], batch["coords"])
                loss = m1_loss_fn(outputs["logits"], batch["labels"])
        else:
            outputs = model(batch["tile_images"], batch["coords"])
            loss = m1_loss_fn(outputs["logits"], batch["labels"])

        if training:
            loss.backward()
            if GRAD_CLIP_NORM is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

        total_loss += float(loss.detach().cpu())
        total_batches += 1
        all_logits.append(outputs["logits"].detach().cpu().numpy()[0])
        all_labels.append(batch["labels"].detach().cpu().numpy()[0])
        all_times.append(float(batch["os_time_days"].detach().cpu().item()))
        all_events.append(int(batch["os_event"].detach().cpu().item()))

        running_loss = total_loss / max(total_batches, 1)
        running_metrics = compute_epoch_metrics(all_logits, all_times, all_events, all_labels)
        progress.set_postfix({
            "avg_loss": f"{running_loss:.4f}",
            "avg_acc": "nan" if np.isnan(running_metrics["accuracy"]) else f"{running_metrics['accuracy']:.3f}",
            "mean_auc": "nan" if np.isnan(running_metrics.get("mean_auroc", np.nan)) else f"{running_metrics['mean_auroc']:.3f}",
            "last_loss": f"{float(loss.detach().cpu()):.4f}",
            "tiles": batch["n_tiles"],
        })

        del batch, outputs, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metrics = {"loss": total_loss / max(total_batches, 1)}
    metrics.update(compute_epoch_metrics(all_logits, all_times, all_events, all_labels))
    return metrics


def get_monitor_score(metrics: dict, prefix: str = "valid") -> float:
    key = MONITOR_METRIC.replace(f"{prefix}_", "")
    return float(metrics.get(key, np.nan))


def save_checkpoint(path: Path, epoch: int, metrics: dict, is_best: bool) -> None:
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": metrics,
        "training_config": training_config,
        "is_best": is_best,
    }, path)


def plot_training_history(history: list[dict]) -> None:
    hist_df = pd.DataFrame(history)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0, 0].plot(hist_df["epoch"], hist_df["train_loss"], label="train")
    axes[0, 0].plot(hist_df["epoch"], hist_df["valid_loss"], label="valid")
    axes[0, 0].set_title("Multi-horizon BCE Loss")
    axes[0, 0].set_xlabel("epoch")
    axes[0, 0].legend()

    axes[0, 1].plot(hist_df["epoch"], hist_df["train_accuracy"], label="train")
    axes[0, 1].plot(hist_df["epoch"], hist_df["valid_accuracy"], label="valid")
    axes[0, 1].set_title("Mean Horizon Accuracy")
    axes[0, 1].set_xlabel("epoch")
    axes[0, 1].legend()

    axes[1, 0].plot(hist_df["epoch"], hist_df["train_mean_auroc"], label="train AUROC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["valid_mean_auroc"], label="valid AUROC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["train_mean_auprc"], "--", label="train AUPRC")
    axes[1, 0].plot(hist_df["epoch"], hist_df["valid_mean_auprc"], "--", label="valid AUPRC")
    axes[1, 0].set_title("Mean Horizon AUROC/AUPRC")
    axes[1, 0].set_xlabel("epoch")
    axes[1, 0].legend()

    axes[1, 1].plot(hist_df["epoch"], hist_df["lr"], label="lr")
    axes[1, 1].set_title("Learning Rate")
    axes[1, 1].set_xlabel("epoch")
    axes[1, 1].set_yscale("log")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()
    display(hist_df.tail(10))


history = []
best_score = -np.inf if MONITOR_MODE == "max" else np.inf
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_one_epoch(train_dataset, training=True, epoch=epoch)
    valid_metrics = run_one_epoch(valid_dataset, training=False, epoch=epoch)

    monitor_score = get_monitor_score(valid_metrics, prefix="valid")
    scheduler.step(monitor_score)
    current_lr = optimizer.param_groups[0]["lr"]

    if MONITOR_MODE == "max":
        improved = monitor_score > best_score + MIN_DELTA
    else:
        improved = monitor_score < best_score - MIN_DELTA

    if improved:
        best_score = monitor_score
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "valid_loss": valid_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "valid_accuracy": valid_metrics["accuracy"],
        "train_c_index": train_metrics["c_index"],
        "valid_c_index": valid_metrics["c_index"],
        "train_mean_auroc": train_metrics.get("mean_auroc", np.nan),
        "valid_mean_auroc": valid_metrics.get("mean_auroc", np.nan),
        "train_mean_auprc": train_metrics.get("mean_auprc", np.nan),
        "valid_mean_auprc": valid_metrics.get("mean_auprc", np.nan),
        "lr": current_lr,
    }
    for name in HORIZON_NAMES:
        row[f"train_{name}_auroc"] = train_metrics.get(f"{name}_auroc", np.nan)
        row[f"valid_{name}_auroc"] = valid_metrics.get(f"{name}_auroc", np.nan)
        row[f"train_{name}_auprc"] = train_metrics.get(f"{name}_auprc", np.nan)
        row[f"valid_{name}_auprc"] = valid_metrics.get(f"{name}_auprc", np.nan)
        row[f"train_{name}_accuracy"] = train_metrics.get(f"{name}_accuracy", np.nan)
        row[f"valid_{name}_accuracy"] = valid_metrics.get(f"{name}_accuracy", np.nan)
    history.append(row)

    log_df = pd.DataFrame(history)
    log_df.to_csv(TRAIN_LOG_PATH, index=False)

    checkpoint_metrics = {"train": train_metrics, "valid": valid_metrics, "monitor_score": monitor_score}
    save_checkpoint(LAST_CHECKPOINT_PATH, epoch, checkpoint_metrics, is_best=False)
    if improved:
        save_checkpoint(BEST_CHECKPOINT_PATH, epoch, checkpoint_metrics, is_best=True)

    status = "best saved" if improved else f"no improve {epochs_without_improvement}/{PATIENCE}"
    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_metrics['loss']:.4f} valid_loss={valid_metrics['loss']:.4f} | "
        f"train_acc={train_metrics['accuracy']:.3f} valid_acc={valid_metrics['accuracy']:.3f} | "
        f"valid_AUROC={valid_metrics.get('mean_auroc', float('nan')):.3f} "
        f"valid_AUPRC={valid_metrics.get('mean_auprc', float('nan')):.3f} | "
        f"lr={current_lr:.2e} | best_epoch={best_epoch} ({status})"
    )

    if epoch % LOG_EVERY_EPOCHS == 0 or improved:
        plot_training_history(history)

    if epoch % SAVE_EVERY_EPOCHS == 0:
        save_checkpoint(M1_MODEL_DIR / f"checkpoint_epoch_{epoch:03d}.pt", epoch, checkpoint_metrics, is_best=False)

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}. Best epoch={best_epoch}, best {MONITOR_METRIC}={best_score:.4f}")
        break

print("training finished")
print("best_epoch:", best_epoch)
print("best_score:", best_score)
print("best checkpoint:", BEST_CHECKPOINT_PATH)
